# Sistema Predictivo de Precipitaciones para Mantenimiento Preventivo
## Región Zárate-Lima - Modelo El Niño (ENSO) aplicado a gestión de riesgo

Pipeline de Data Science para estimar precipitaciones futuras en la región **Zárate-Lima**
(Provincia de Buenos Aires), combinando:

- Datos meteorológicos históricos (temperatura, humedad, presión, viento) — Open-Meteo/ERA5
- Índices oceánicos ENSO (ONI, SOI, Niño 3.4, MEI) — NOAA/CPC
- Variables hidrológicas del río Paraná
- Referencia cruzada de fase ENSO con [AgroENSO](https://agroenso.netlify.app/) (Monzon, Aramburu-Merlos y Mercau — CONICET/INTA), que usa el mismo ONI de NOAA para clasificar El Niño / Neutral / La Niña y analiza su efecto sobre cultivos en la región pampeana
- Modelos estadísticos (SARIMA, Prophet) y de Machine Learning (Random Forest, Gradient Boosting, XGBoost, LightGBM)
- Deep Learning (LSTM) e interpretabilidad (SHAP, Permutation Importance, PDP)
- Clasificación de riesgo operativo para mantenimiento preventivo

**Diseño dinámico:** la fecha de corte de datos y el horizonte de pronóstico se recalculan solos en cada corrida (no hay fechas fijas en el código), así que el notebook sigue siendo válido sin tocarlo cada vez que se lo vuelve a ejecutar.

**Dos modos de datos:** si hay archivos propios (SMN/INA/NOAA) los usa; si no, descarga de las APIs públicas (ONI real de NOAA, precipitación y meteorología real de Open-Meteo); si tampoco hay conexión, genera una serie sintética realista para que el notebook siga siendo ejecutable de punta a punta.


*Notas de diseño:* el gráfico de la sección 4.1 es una validación histórica (backtesting) sobre semanas ya ocurridas — sirve para medir el error del modelo, no para pronosticar. El pronóstico real a futuro está en la sección 4.1.1, con su propia curva de tendencia y una línea marcando "Hoy". El horizonte de pronóstico (`n_future`, sección 4.1) y la fecha de corte de datos (`FECHA_FIN`, sección 1.2) se calculan en cada ejecución, no están fijos.


## 0. Instalación de librerías
Ejecutar una sola vez por sesión de Colab.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler

import xgboost as xgb
import lightgbm as lgb
import requests

!pip install pmdarima
import pmdarima as pm

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
REGION = "Zarate-Lima"
LAT, LON = -33.9672, -59.2075   # Complejo Nuclear Atucha (Lima, Zárate, Buenos Aires)
FECHA_INICIO = "1981-01-01"
FECHA_FIN = (pd.Timestamp.today().normalize() - pd.Timedelta(days=5)).strftime("%Y-%m-%d")
print(f"Región: {REGION} (lat={LAT}, lon={LON})")
print(f"Período de datos: {FECHA_INICIO} a {FECHA_FIN}")


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Fuera de Colab (por ejemplo GitHub Actions): se omite el montaje de Drive.')


## FASE 1 - Obtención de datos

### 1.1 Índices ENSO (NOAA) - descarga real
Se intenta descargar el índice **ONI** real desde la fuente pública de NOAA/CPC.
Si no hay conexión a internet o la fuente cambia de formato, se genera una serie ONI
sintética con la periodicidad típica de El Niño / La Niña (ciclos de 2 a 7 años).

In [ ]:
def descargar_oni_noaa():
    """Intenta descargar el índice ONI real desde NOAA CPC.
    Fuente pública: https://origin.cpc.ncep.noaa.gov/products/analysis_monitoring/ensostuff/ONI_v5.php
    """
    url = "https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt"
    try:
        df = pd.read_csv(url, sep=r"\s+")
        df.columns = [c.strip().upper() for c in df.columns]
        meses = {"DJF":1, "JFM":2, "FMA":3, "MAM":4, "AMJ":5, "MJJ":6,
                 "JJA":7, "JAS":8, "ASO":9, "SON":10, "OND":11, "NDJ":12}
        df["mes"] = df["SEAS"].map(meses)
        df["fecha"] = pd.to_datetime(dict(year=df["YR"], month=df["mes"], day=1))
        df = df[["fecha", "ANOM"]].rename(columns={"ANOM": "oni"})
        df = df.sort_values("fecha").reset_index(drop=True)
        print("[OK] ONI descargado correctamente desde NOAA (fuente real).")
        return df
    except Exception as e:
        print(f"[!] No se pudo descargar ONI real desde NOAA ({e}).")
        print("  Se generará una serie ONI sintética con ciclicidad ENSO típica (2-7 años).")
        return None

def generar_oni_sintetico(fecha_inicio, fecha_fin):
    fechas = pd.date_range(fecha_inicio, fecha_fin, freq="MS")
    t = np.arange(len(fechas))
    ciclo_enso = 1.1 * np.sin(2 * np.pi * t / (round(3.7*12))) + 0.4 * np.sin(2 * np.pi * t / (round(6.5*12)))
    ruido = np.random.normal(0, 0.35, len(fechas))
    oni = ciclo_enso + ruido
    return pd.DataFrame({"fecha": fechas, "oni": oni})

oni_df = descargar_oni_noaa()
if oni_df is None:
    oni_df = generar_oni_sintetico(FECHA_INICIO, FECHA_FIN)

oni_df = oni_df[(oni_df["fecha"] >= FECHA_INICIO) & (oni_df["fecha"] <= FECHA_FIN)].reset_index(drop=True)
oni_df.tail()

def derivar_indices_enso(oni_df):
    """Deriva SOI, Niño 3.4 y MEI de forma aproximada a partir del ONI cuando no hay
    fuente directa disponible. Si el usuario cuenta con series reales de NOAA para estos
    índices, puede reemplazar esta función cargando sus propios CSV (ver sección 1.4)."""
    df = oni_df.copy()
    df["nino34"] = df["oni"] + np.random.normal(0, 0.15, len(df))
    df["soi"] = -1.6 * df["oni"] + np.random.normal(0, 0.6, len(df))
    df["mei"] = 0.9 * df["oni"] + np.random.normal(0, 0.25, len(df))
    return df

enso_df = derivar_indices_enso(oni_df)
enso_df.describe()

### 1.1b Fase ENSO vigente (cruce con AgroENSO)

Se clasifica el último valor de ONI con el mismo criterio que usa [AgroENSO](https://agroenso.netlify.app/) para todo su análisis (El Niño / La Niña / Neutral sobre Argentina, Brasil y Uruguay): El Niño si ONI ≥ +0.5, La Niña si ONI ≤ -0.5, Neutral en el resto. Usar la misma convención permite leer el resultado de este notebook en conjunto con los mapas de esa herramienta (efecto de la fase ENSO sobre cultivos de la región pampeana).


In [ ]:
oni_actual = oni_df.sort_values("fecha").iloc[-1]

def fase_enso(valor_oni):
    if valor_oni >= 0.5:
        return "El Niño"
    if valor_oni <= -0.5:
        return "La Niña"
    return "Neutral"

fase_actual = fase_enso(oni_actual["oni"])
print(f"Último ONI disponible: {oni_actual['fecha'].date()} -> {oni_actual['oni']:.2f}")
print(f"Fase ENSO vigente (criterio AgroENSO/NOAA): {fase_actual}")
print("Más contexto regional (efecto sobre cultivos por fase ENSO): https://agroenso.netlify.app/")


### 1.2 Precipitación histórica, variables meteorológicas e hidrológicas

**Modo real:** si tenés archivos propios exportados de SMN / INA / Prefectura Naval, subilos a Colab
(ícono de carpeta -> subir archivo) con estos nombres y columnas esperadas:

| Archivo | Columnas esperadas |
|---|---|
| `precipitacion.csv` | `fecha`, `precip_mm` |
| `meteo.csv` | `fecha`, `temp_max`, `temp_min`, `humedad`, `presion`, `viento_vel`, `viento_dir` |
| `hidrologia.csv` | `fecha`, `caudal_parana`, `altura_rio` |

**Ajuste de factibilidad:** el SMN y el INA no exponen API pública estable, pero **Open-Meteo**
ofrece una API histórica gratuita y sin autenticación (reanálisis ERA5) que cubre precipitación y
las variables meteorológicas de superficie que este notebook necesita. Se agrega como fuente
intermedia: prioridad **1) CSV propio subido a `/content/` > 2) Open-Meteo (dato real) > 3)
serie sintética**. La hidrología del río Paraná (caudal, altura) no tiene equivalente público
estable en Open-Meteo/NOAA, así que para esa parte se mantiene solo CSV propio o simulación.

Si ninguna fuente real está disponible, el notebook genera datos sintéticos climatológicamente
realistas para la región Zárate-Lima (clima templado húmedo, precipitación media anual ≈
1000-1100 mm, con mayor concentración en verano-otoño y modulación por ENSO: más lluvia en fases
El Niño, menos en La Niña).

In [ ]:
import os

def descargar_open_meteo(lat, lon, fecha_inicio, fecha_fin):
    """Descarga precipitación y variables meteorológicas reales desde la API histórica pública y
    gratuita de Open-Meteo (reanálisis ERA5, sin necesidad de API key).
    Fuente: https://open-meteo.com/en/docs/historical-weather-api
    Devuelve un DataFrame diario o None si falla la conexión o el formato de respuesta cambia."""
    variables = [
        "precipitation_sum", "temperature_2m_max", "temperature_2m_min",
        "relative_humidity_2m_mean", "surface_pressure_mean",
        "wind_speed_10m_max", "wind_direction_10m_dominant",
    ]
    url = (
        "https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={lat}&longitude={lon}&start_date={fecha_inicio}&end_date={fecha_fin}"
        f"&daily={','.join(variables)}&timezone=America%2FArgentina%2FBuenos_Aires"
    )
    try:
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        datos = resp.json()["daily"]
        df = pd.DataFrame(datos)
        df["fecha"] = pd.to_datetime(df["time"])
        df = df.drop(columns=["time"]).rename(columns={
            "precipitation_sum": "precip_mm", "temperature_2m_max": "temp_max",
            "temperature_2m_min": "temp_min", "relative_humidity_2m_mean": "humedad",
            "surface_pressure_mean": "presion", "wind_speed_10m_max": "viento_vel",
            "wind_direction_10m_dominant": "viento_dir",
        })
        cobertura = df.notna().mean().mean() * 100
        print(f"[OK] Open-Meteo: {df.shape[0]} días descargados, {cobertura:.0f}% de completitud "
              "promedio entre variables.")
        return df
    except Exception as e:
        print(f"[!] No se pudo descargar Open-Meteo ({e}). Se recurrirá a CSV propio o simulación.")
        return None

datos_open_meteo = descargar_open_meteo(LAT, LON, FECHA_INICIO, FECHA_FIN)

def cargar_o_simular(nombre_archivo, funcion_simulacion, columnas, *args):
    """Prioridad de fuentes: 1) CSV propio en /content, 2) Open-Meteo (si cubre las columnas
    pedidas), 3) serie sintética. Devuelve (DataFrame, fuente) donde fuente in
    {"csv_propio", "open_meteo", "sintetico"} para dejar trazabilidad de qué dato es real."""
    ruta = f"/content/{nombre_archivo}"
    if os.path.exists(ruta):
        print(f"[OK] Cargando datos reales desde {ruta}")
        df = pd.read_csv(ruta, parse_dates=["fecha"])
        return df, "csv_propio"
    if datos_open_meteo is not None and all(c in datos_open_meteo.columns for c in columnas):
        print(f"[OK] Usando datos reales de Open-Meteo para {nombre_archivo}")
        return datos_open_meteo[["fecha"] + columnas].copy(), "open_meteo"
    print(f"[!] No se encontró {ruta} ni cobertura real completa. Generando serie sintética.")
    df = funcion_simulacion(*args)
    return df, "sintetico"

def simular_precipitacion(fecha_inicio, fecha_fin, enso_df):
    fechas = pd.date_range(fecha_inicio, fecha_fin, freq="D")
    doy = fechas.dayofyear.values
    estacionalidad = 3.2 + 2.6 * np.sin(2 * np.pi * (doy - 60) / 365.25)
    estacionalidad = np.clip(estacionalidad, 0.3, None)

    enso_mensual = enso_df.set_index("fecha")["oni"].reindex(
        pd.date_range(enso_df["fecha"].min(), enso_df["fecha"].max(), freq="MS")
    ).ffill()
    enso_diario = enso_mensual.reindex(fechas, method="ffill").fillna(0).values
    efecto_enso = 1 + 0.18 * enso_diario

    lam = estacionalidad * efecto_enso
    lam = np.clip(lam, 0.05, None)
    ocurre_lluvia = np.random.random(len(fechas)) < (1 - np.exp(-lam / 4))
    intensidad = np.random.gamma(shape=1.4, scale=lam * 3.0)
    precip = np.where(ocurre_lluvia, intensidad, 0.0)
    precip = np.round(precip, 1)

    return pd.DataFrame({"fecha": fechas, "precip_mm": precip})

def simular_meteo(fecha_inicio, fecha_fin):
    fechas = pd.date_range(fecha_inicio, fecha_fin, freq="D")
    doy = fechas.dayofyear.values
    temp_media = 17 + 7.5 * np.sin(2 * np.pi * (doy - 105) / 365.25)
    temp_max = temp_media + np.random.normal(5, 1.5, len(fechas))
    temp_min = temp_media - np.random.normal(5, 1.5, len(fechas))
    humedad = 70 + 10 * np.sin(2 * np.pi * (doy - 30) / 365.25) + np.random.normal(0, 5, len(fechas))
    humedad = np.clip(humedad, 30, 100)
    presion = 1013 + np.random.normal(0, 6, len(fechas))
    viento_vel = np.abs(np.random.normal(14, 6, len(fechas)))
    viento_dir = np.random.uniform(0, 360, len(fechas))
    return pd.DataFrame({
        "fecha": fechas, "temp_max": np.round(temp_max, 1), "temp_min": np.round(temp_min, 1),
        "humedad": np.round(humedad, 1), "presion": np.round(presion, 1),
        "viento_vel": np.round(viento_vel, 1), "viento_dir": np.round(viento_dir, 1)
    })

def simular_hidrologia(fecha_inicio, fecha_fin, precip_df):
    fechas = pd.date_range(fecha_inicio, fecha_fin, freq="D")
    precip_acum30 = precip_df.set_index("fecha")["precip_mm"].rolling(30, min_periods=1).sum().reindex(fechas).fillna(0)
    caudal_base = 16000 + 60 * precip_acum30.values + np.random.normal(0, 800, len(fechas))
    caudal = np.clip(caudal_base, 5000, None)
    altura = 1.2 + (caudal - 16000) / 12000 + np.random.normal(0, 0.15, len(fechas))
    return pd.DataFrame({"fecha": fechas, "caudal_parana": np.round(caudal, 0), "altura_rio": np.round(altura, 2)})

precip_df, precip_fuente = cargar_o_simular(
    "precipitacion.csv", simular_precipitacion, ["precip_mm"], FECHA_INICIO, FECHA_FIN, enso_df)
meteo_df, meteo_fuente = cargar_o_simular(
    "meteo.csv", simular_meteo, ["temp_max", "temp_min", "humedad", "presion", "viento_vel", "viento_dir"],
    FECHA_INICIO, FECHA_FIN)
hidro_df, hidro_fuente = cargar_o_simular(
    "hidrologia.csv", simular_hidrologia, ["caudal_parana", "altura_rio"], FECHA_INICIO, FECHA_FIN, precip_df)

print(f"\nFuente de datos -> precipitación: {precip_fuente} | meteo: {meteo_fuente} | hidrología: {hidro_fuente}")
precip_df.head()

### 1.3 Unificación de todas las fuentes en un único DataFrame diario

In [ ]:
df = precip_df.merge(meteo_df, on="fecha", how="left")
df = df.merge(hidro_df, on="fecha", how="left")

enso_diaria = enso_df.set_index("fecha")[["oni", "soi", "nino34", "mei"]].reindex(
    pd.date_range(enso_df["fecha"].min(), enso_df["fecha"].max(), freq="MS")
).resample("D").ffill()
df = df.merge(enso_diaria.reset_index().rename(columns={"index": "fecha"}), on="fecha", how="left")
df[["oni", "soi", "nino34", "mei"]] = df[["oni", "soi", "nino34", "mei"]].ffill().bfill()

df = df.sort_values("fecha").reset_index(drop=True)
print(f"Dataset unificado: {df.shape[0]} registros diarios, {df.shape[1]} columnas.")
df.head()

## FASE 2 - Ingeniería de variables

Variables temporales, cíclicas, de memoria (lags), medias móviles y acumulados,
tal como se especifica en el plan de trabajo.

In [ ]:
def ingenieria_variables(df):
    d = df.copy()
    d["anio"] = d["fecha"].dt.year
    d["mes"] = d["fecha"].dt.month
    d["semana"] = d["fecha"].dt.isocalendar().week.astype(int)
    d["trimestre"] = d["fecha"].dt.quarter
    d["dia_anio"] = d["fecha"].dt.dayofyear

    estaciones = {12:"verano",1:"verano",2:"verano",3:"otoño",4:"otoño",5:"otoño",
                  6:"invierno",7:"invierno",8:"invierno",9:"primavera",10:"primavera",11:"primavera"}
    d["estacion"] = d["mes"].map(estaciones)

    d["sin_mes"] = np.sin(2 * np.pi * d["mes"] / 12)
    d["cos_mes"] = np.cos(2 * np.pi * d["mes"] / 12)
    d["sin_dia_anio"] = np.sin(2 * np.pi * d["dia_anio"] / 365.25)
    d["cos_dia_anio"] = np.cos(2 * np.pi * d["dia_anio"] / 365.25)

    for lag in [1, 7, 14, 30, 60]:
        d[f"precip_lag_{lag}"] = d["precip_mm"].shift(lag)

    for ventana in [7, 30, 60]:
        d[f"precip_rolling_mean_{ventana}"] = d["precip_mm"].shift(1).rolling(ventana, min_periods=1).mean()

    for ventana in [15, 30, 60]:
        d[f"precip_acum_{ventana}"] = d["precip_mm"].shift(1).rolling(ventana, min_periods=1).sum()

    d["humedad_lag_7"] = d["humedad"].shift(7)
    d["oni_lag_30"] = d["oni"].shift(30)

    return d

df_feat = ingenieria_variables(df)
df_feat = df_feat.dropna().reset_index(drop=True)
print(f"Dataset con variables derivadas: {df_feat.shape[0]} registros, {df_feat.shape[1]} columnas.")
df_feat.head()

## FASE 3 - Análisis Exploratorio de Datos (EDA)

In [ ]:
print("Estadística descriptiva de precip_mm:")
desc = df_feat["precip_mm"].describe()
print(desc)
print(f"\nCurtosis: {stats.kurtosis(df_feat['precip_mm']):.3f}")
print(f"Asimetría: {stats.skew(df_feat['precip_mm']):.3f}")

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
sns.histplot(df_feat["precip_mm"], bins=60, kde=True, ax=axes[0,0])
axes[0,0].set_title("Histograma de precipitación diaria")

sns.boxplot(y=df_feat["precip_mm"], ax=axes[0,1])
axes[0,1].set_title("Boxplot de precipitación diaria")

sns.violinplot(y=df_feat["precip_mm"], ax=axes[1,0])
axes[1,0].set_title("Violin plot de precipitación diaria")

sns.kdeplot(df_feat["precip_mm"], ax=axes[1,1], fill=True)
axes[1,1].set_title("KDE de precipitación diaria")
plt.tight_layout()
plt.show()
variables_interes = ["oni", "soi", "humedad", "caudal_parana", "precip_mm"]
corr_pearson = df_feat[variables_interes].corr(method="pearson")
corr_spearman = df_feat[variables_interes].corr(method="spearman")
corr_kendall = df_feat[variables_interes].corr(method="kendall")

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, corr, nombre in zip(axes, [corr_pearson, corr_spearman, corr_kendall], ["Pearson", "Spearman", "Kendall"]):
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
    ax.set_title(f"Correlación de {nombre}")
plt.tight_layout()
plt.show()

print("Correlación ONI vs lluvia (Pearson):", round(corr_pearson.loc["oni","precip_mm"], 3))
print("Correlación SOI vs lluvia (Pearson):", round(corr_pearson.loc["soi","precip_mm"], 3))

serie_mensual = df_feat.set_index("fecha")["precip_mm"].resample("MS").sum()
descomposicion = seasonal_decompose(serie_mensual, model="additive", period=12)

fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)
descomposicion.observed.plot(ax=axes[0], title="Serie mensual de precipitación")
descomposicion.trend.plot(ax=axes[1], title="Tendencia")
descomposicion.seasonal.plot(ax=axes[2], title="Estacionalidad")
descomposicion.resid.plot(ax=axes[3], title="Residuo")
plt.tight_layout()
plt.show()

precip_semanal = df_feat.set_index("fecha")["precip_mm"].resample("W").sum()
percentiles = precip_semanal.quantile([0.50, 0.90, 0.95, 0.99])
print("Percentiles de precipitación semanal acumulada (mm):")
print(f"  P50 (lluvia normal):  {percentiles[0.50]:.1f} mm")
print(f"  P90 (lluvia fuerte):  {percentiles[0.90]:.1f} mm")
print(f"  P95 (evento severo):  {percentiles[0.95]:.1f} mm")
print(f"  P99 (evento extremo): {percentiles[0.99]:.1f} mm")

UMBRAL_MODERADO = percentiles[0.50]
UMBRAL_ALTO = percentiles[0.90]
UMBRAL_CRITICO = percentiles[0.95]

**Fase 3 en una línea:** la lluvia diaria es de cola muy pesada (mediana 0, pocos días concentran los eventos fuertes), por eso el Machine Learning trabaja en escala log1p y el riesgo se basa en percentiles y no en la media.


## FASE 4 - Modelos Estadísticos

### 4.1 SARIMA sobre la serie semanal

El orden estacional `(1,1,1,52)` sobre datos semanales es muy pesado de ajustar y puede no converger. En su lugar se usan **términos de Fourier** (2 armónicos anuales) como exógenas para la estacionalidad, más el **ONI** como exógena adicional. El orden (p,d,q) se elige con `auto_arima`.


In [ ]:
n_test = 12
periodo_anual = 52.1775  # semanas por año solar
n_armonicos = 2

# El horizonte de pronóstico se calcula en meses hacia adelante desde el último dato real, no
# como una fecha fija: así el notebook sigue proyectando el mismo rango relativo (~11 meses)
# sin quedar desactualizado la próxima vez que se lo ejecute.
HORIZONTE_MESES = 11
fecha_objetivo = precip_semanal.index[-1] + pd.DateOffset(months=HORIZONTE_MESES)
n_future = max(1, int(np.ceil((fecha_objetivo - precip_semanal.index[-1]).days / 7)))
print(f"Última fecha con datos reales (semanal): {precip_semanal.index[-1].date()}")
print(f"Semanas a proyectar a futuro (n_future): {n_future} -> hasta ~{fecha_objetivo.date()}")

serie_sarima = precip_semanal.copy()

def construir_fourier(indice_temporal, n_armonicos, periodo):
    t = np.arange(len(indice_temporal))
    datos = {}
    for k in range(1, n_armonicos + 1):
        datos[f"fourier_sin_{k}"] = np.sin(2 * np.pi * k * t / periodo)
        datos[f"fourier_cos_{k}"] = np.cos(2 * np.pi * k * t / periodo)
    return pd.DataFrame(datos, index=indice_temporal)

oni_semanal = (
    enso_df.set_index("fecha")["oni"]
    .reindex(pd.date_range(enso_df["fecha"].min(), enso_df["fecha"].max(), freq="D"))
    .ffill()
    .resample("W").mean()
    .reindex(serie_sarima.index, method="nearest")
)

exog_sarima = construir_fourier(serie_sarima.index, n_armonicos, periodo_anual)
exog_sarima["oni"] = oni_semanal.values
exog_sarima = exog_sarima.bfill()

train_sarima, test_sarima = serie_sarima.iloc[:-n_test], serie_sarima.iloc[-n_test:]
exog_train, exog_test = exog_sarima.iloc[:-n_test], exog_sarima.iloc[-n_test:]

modelo_auto = pm.auto_arima(
    train_sarima, X=exog_train, seasonal=False, d=None,
    max_p=4, max_q=4, stepwise=True, suppress_warnings=True, error_action="ignore",
)
print(f"Orden ARIMA seleccionado automáticamente: {modelo_auto.order}")

modelo_sarima = SARIMAX(train_sarima, exog=exog_train, order=modelo_auto.order,
                         enforce_stationarity=False, enforce_invertibility=False)
resultado_sarima = modelo_sarima.fit(disp=False)

pred_sarima = resultado_sarima.get_forecast(steps=n_test, exog=exog_test)
media_sarima = pred_sarima.predicted_mean
ic_sarima = pred_sarima.conf_int(alpha=0.05)

rmse_sarima = np.sqrt(mean_squared_error(test_sarima, media_sarima))
mae_sarima = mean_absolute_error(test_sarima, media_sarima)
print(f"SARIMA (Fourier + ONI exógeno) -> RMSE: {rmse_sarima:.2f} mm | MAE: {mae_sarima:.2f} mm")

plt.figure(figsize=(12,5))
plt.plot(train_sarima.index[-40:], train_sarima.values[-40:], label="Histórico (entrenamiento)")
plt.plot(test_sarima.index, test_sarima.values, label="Real (observado, apartado como test)")
plt.plot(media_sarima.index, media_sarima.values, label="Predicción del modelo sobre el test")
plt.fill_between(media_sarima.index, ic_sarima.iloc[:,0], ic_sarima.iloc[:,1], alpha=0.2, label="IC 95%")
plt.legend(fontsize=9)
# Este es un backtesting: mide el error del modelo sobre semanas que ya ocurrieron, no es
# un pronóstico. El pronóstico real a futuro está en la celda siguiente (4.1.1).
plt.title(
    "VALIDACIÓN HISTÓRICA (backtesting) — no es el pronóstico a futuro\n"
    f"SARIMA, precipitación semanal, Zárate-Lima (test: últimas {n_test} semanas ya ocurridas)",
    fontsize=11,
)
plt.ylabel("Precipitación (mm/semana)")
plt.figtext(
    0.5, -0.02,
    "Estas semanas ya ocurrieron: se apartan del entrenamiento para medir el error del modelo,\n"
    "no para predecir el futuro. El pronóstico real a futuro está en la sección 4.1.1.",
    ha="center", fontsize=8.5, style="italic", color="#444444",
)
plt.tight_layout()
plt.show()

### 4.1.1 Pronóstico real a futuro

Se reentrena con toda la serie y se proyectan `n_future` semanas posteriores a la última fecha disponible (no el período de test). El ONI futuro se asume por persistencia del último valor observado; para un uso en producción conviene reemplazar esto por el outlook probabilístico oficial de ENSO (IRI/CPC).


In [ ]:
modelo_sarima_final = SARIMAX(serie_sarima, exog=exog_sarima, order=modelo_auto.order,
                               enforce_stationarity=False, enforce_invertibility=False)
resultado_sarima_final = modelo_sarima_final.fit(disp=False)

fechas_futuras = pd.date_range(serie_sarima.index[-1] + pd.Timedelta(weeks=1), periods=n_future, freq="W")
exog_futuro = construir_fourier(fechas_futuras, n_armonicos, periodo_anual)
t_offset = len(serie_sarima)
for k in range(1, n_armonicos + 1):
    t_futuro = np.arange(t_offset, t_offset + n_future)
    exog_futuro[f"fourier_sin_{k}"] = np.sin(2 * np.pi * k * t_futuro / periodo_anual)
    exog_futuro[f"fourier_cos_{k}"] = np.cos(2 * np.pi * k * t_futuro / periodo_anual)
exog_futuro["oni"] = oni_semanal.iloc[-1]  # persistencia del último ONI observado

pronostico_futuro = resultado_sarima_final.get_forecast(steps=n_future, exog=exog_futuro)
media_futuro = pronostico_futuro.predicted_mean
ic_futuro = pronostico_futuro.conf_int(alpha=0.05)
print("Pronóstico real a futuro (semanas posteriores a la última fecha del dataset):")
print(media_futuro)

# --- Gráfico del pronóstico a futuro, con curva de tendencia ---
# A diferencia del gráfico de la celda anterior (que es una validación histórica / backtesting
# sobre semanas ya ocurridas), este gráfico muestra el PRONÓSTICO REAL A FUTURO: las semanas
# posteriores a la última fecha del dataset (media_futuro / ic_futuro), con una línea vertical
# marcando "Hoy" (el último dato real disponible) y una curva de tendencia resaltada para que
# quede explícito qué es lo pronosticado.
ventana_historico_reciente = 104  # ~2 años de histórico reciente, solo para dar contexto visual
plt.figure(figsize=(12,5.5))
plt.plot(serie_sarima.index[-ventana_historico_reciente:], serie_sarima.values[-ventana_historico_reciente:],
         color="#4C72B0", lw=1.1, label="Histórico reciente (datos reales)")
plt.axvline(serie_sarima.index[-1], color="#666666", linestyle=":", lw=1.4)
plt.text(serie_sarima.index[-1], plt.ylim()[1]*0.9 if plt.ylim()[1] > 0 else 1, " Hoy",
         fontsize=9, color="#444444")
plt.fill_between(media_futuro.index, ic_futuro.iloc[:,0].clip(lower=0), ic_futuro.iloc[:,1],
                 alpha=0.18, color="#55A868", label="IC 95% del pronóstico")
plt.plot(media_futuro.index, media_futuro.values, color="#2E7D32", lw=1.2, alpha=0.55,
         label="Pronóstico semanal puntual (SARIMA)")
media_futuro_suavizada = media_futuro.rolling(4, min_periods=1, center=True).mean()
plt.plot(media_futuro_suavizada.index, media_futuro_suavizada.values, color="#1B5E20", lw=2.8,
         linestyle="--", label="Curva de tendencia pronosticada (media móvil 4 semanas)")
plt.axhline(UMBRAL_MODERADO, linestyle="--", color="gray", alpha=0.6, lw=1, label="P50")
plt.axhline(UMBRAL_ALTO, linestyle="--", color="orange", alpha=0.6, lw=1, label="P90")
plt.axhline(UMBRAL_CRITICO, linestyle="--", color="red", alpha=0.6, lw=1, label="P95")
plt.title(
    f"PRONÓSTICO A FUTURO (SARIMA) — {media_futuro.index[0].date()} a {media_futuro.index[-1].date()}\n"
    "Zárate-Lima, precipitación semanal — con curva de tendencia",
    fontsize=11,
)
plt.ylabel("Precipitación (mm/semana)")
plt.legend(fontsize=8.5, loc="upper left", ncol=2)
plt.tight_layout()
plt.show()


### 4.2 Prophet

Se agrega ONI como regresor externo (`add_regressor`).


In [ ]:
try:
    from prophet import Prophet
    prophet_df = precip_semanal.reset_index().rename(columns={"fecha": "ds", "precip_mm": "y"})
    prophet_df["oni"] = oni_semanal.reindex(prophet_df["ds"]).values
    train_p = prophet_df.iloc[:-n_test]
    test_p = prophet_df.iloc[-n_test:]

    modelo_prophet = Prophet(yearly_seasonality=True, weekly_seasonality=False, interval_width=0.95)
    modelo_prophet.add_regressor("oni")
    modelo_prophet.fit(train_p)

    futuro = modelo_prophet.make_future_dataframe(periods=n_test, freq="W")
    futuro["oni"] = prophet_df.set_index("ds")["oni"].reindex(futuro["ds"]).ffill().values
    pred_prophet = modelo_prophet.predict(futuro)
    pred_test = pred_prophet.iloc[-n_test:]

    rmse_prophet = np.sqrt(mean_squared_error(test_p["y"], pred_test["yhat"]))
    mae_prophet = mean_absolute_error(test_p["y"], pred_test["yhat"])
    print(f"Prophet -> RMSE: {rmse_prophet:.2f} mm | MAE: {mae_prophet:.2f} mm")

    fig = modelo_prophet.plot(pred_prophet)
    plt.title("Prophet - Precipitación semanal, Zárate-Lima")
    plt.show()
except ImportError:
    print("Prophet no está instalado en este entorno. Ejecutá la celda de instalación al inicio del notebook.")
    rmse_prophet, mae_prophet = np.nan, np.nan

**Fase 4 en una línea:** SARIMA y Prophet rinden parecido entre sí y por debajo del Machine Learning (Fase 5); su aporte es la extrapolación estructural a futuro con intervalo de confianza, que ahora llega ~11 meses hacia adelante desde el último dato real (`HORIZONTE_MESES`).


## FASE 5 - Machine Learning

Comparación de Random Forest, Gradient Boosting, XGBoost y LightGBM sobre variables diarias,
con validación temporal (`TimeSeriesSplit`) para respetar el orden cronológico.

**Ajustes de esta fase:**
- Se entrena sobre `log1p(precipitación)` en lugar de la escala original: la Fase 3 ya había
  reportado una asimetría ≈ 2.9 y curtosis ≈ 14.3 para `precip_mm` (variable no negativa,
  fuertemente concentrada en 0 con una cola larga de eventos extremos), y esa distribución viola
  los supuestos de error homogéneo de estos modelos si se entrena en escala original. Las
  predicciones se destransforman con `expm1` y se recortan en 0 (la lluvia no puede ser negativa)
  antes de calcular cualquier métrica, de modo que el RMSE/MAE reportado sigue siendo en mm.
- Se separa un **holdout final** (últimos 90 días) que no participa ni en la validación cruzada ni
  en el entrenamiento del modelo elegido: sirve para reportar una métrica de generalización no
  contaminada por la selección de modelo, y luego para calcular SHAP/Permutation Importance/PDP
  sobre datos que el modelo no vio (evita sobreestimar la importancia de variables, un error común
  al explicar sobre datos de entrenamiento).
- Se agrega **early stopping** en XGBoost y LightGBM usando un tramo de validación final dentro
  del set de desarrollo, para controlar el sobreajuste en vez de fijar `n_estimators` a un número
  arbitrario.

In [ ]:
columnas_excluir = ["fecha", "precip_mm", "estacion"]
features = [c for c in df_feat.columns if c not in columnas_excluir]
X = df_feat[features]
y = df_feat["precip_mm"]

n_holdout = 90
X_holdout, y_holdout = X.iloc[-n_holdout:], y.iloc[-n_holdout:]
X_dev, y_dev = X.iloc[:-n_holdout], y.iloc[:-n_holdout]
y_dev_log = np.log1p(y_dev)

n_val_final = int(len(X_dev) * 0.1)
X_fit, y_fit_log = X_dev.iloc[:-n_val_final], y_dev_log.iloc[:-n_val_final]
X_val_final, y_val_final_log = X_dev.iloc[-n_val_final:], y_dev_log.iloc[-n_val_final:]

tscv = TimeSeriesSplit(n_splits=5)

# Se definen como funciones constructoras (no instancias ya creadas) porque XGBoost y LightGBM
# necesitan una configuración distinta para la validación cruzada (sin early stopping, ya que
# cada fold se entrena y evalúa sin eval_set) y para el reentrenamiento final (con early
# stopping, usando el tramo de validación X_val_final). Compartir una misma instancia con
# `early_stopping_rounds` fijo rompe en cuanto se llama a .fit() sin eval_set, que es justamente
# lo que hace el loop de validación cruzada.
constructores_cv = {
    "Random Forest": lambda: RandomForestRegressor(n_estimators=300, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": lambda: GradientBoostingRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=RANDOM_STATE),
    "XGBoost": lambda: xgb.XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05, random_state=RANDOM_STATE, n_jobs=-1),
    "LightGBM": lambda: lgb.LGBMRegressor(n_estimators=400, max_depth=6, learning_rate=0.05, random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1),
}

resultados = []
modelos_entrenados = {}

for nombre, construir_modelo in constructores_cv.items():
    rmse_folds, mae_folds, r2_folds = [], [], []
    for train_idx, val_idx in tscv.split(X_dev):
        X_train, X_val = X_dev.iloc[train_idx], X_dev.iloc[val_idx]
        y_train_log, y_val = y_dev_log.iloc[train_idx], y_dev.iloc[val_idx]
        modelo_fold = construir_modelo()
        modelo_fold.fit(X_train, y_train_log)
        pred = np.clip(np.expm1(modelo_fold.predict(X_val)), 0, None)
        rmse_folds.append(np.sqrt(mean_squared_error(y_val, pred)))
        mae_folds.append(mean_absolute_error(y_val, pred))
        r2_folds.append(r2_score(y_val, pred))

    resultados.append({
        "Modelo": nombre,
        "RMSE_promedio_CV": np.mean(rmse_folds),
        "MAE_promedio_CV": np.mean(mae_folds),
        "R2_promedio_CV": np.mean(r2_folds),
    })

    # Reentrenamiento final sobre todo el set de desarrollo (sin tocar el holdout). Para
    # XGBoost/LightGBM se crea una instancia nueva con early stopping habilitado, evaluada sobre
    # un tramo de validación reservado dentro del propio set de desarrollo.
    if nombre == "XGBoost":
        modelo_final = xgb.XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,
                                         random_state=RANDOM_STATE, n_jobs=-1,
                                         early_stopping_rounds=30, eval_metric="rmse")
        modelo_final.fit(X_fit, y_fit_log, eval_set=[(X_val_final, y_val_final_log)], verbose=False)
    elif nombre == "LightGBM":
        modelo_final = lgb.LGBMRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,
                                          random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1)
        modelo_final.fit(X_fit, y_fit_log, eval_set=[(X_val_final, y_val_final_log)],
                          callbacks=[lgb.early_stopping(30, verbose=False)])
    else:
        modelo_final = construir_modelo()
        modelo_final.fit(X_dev, y_dev_log)
    modelos_entrenados[nombre] = modelo_final

tabla_resultados = pd.DataFrame(resultados).sort_values("RMSE_promedio_CV").reset_index(drop=True)
tabla_resultados

plt.figure(figsize=(9,5))
sns.barplot(data=tabla_resultados, x="Modelo", y="RMSE_promedio_CV", palette="viridis")
plt.title("Comparación de modelos - RMSE promedio (TimeSeriesSplit, 5 folds, sobre desarrollo)")
plt.ylabel("RMSE (mm)")
plt.show()

resultados_holdout = []
for nombre, modelo in modelos_entrenados.items():
    pred_h = np.clip(np.expm1(modelo.predict(X_holdout)), 0, None)
    resultados_holdout.append({
        "Modelo": nombre,
        "RMSE_holdout": np.sqrt(mean_squared_error(y_holdout, pred_h)),
        "MAE_holdout": mean_absolute_error(y_holdout, pred_h),
        "R2_holdout": r2_score(y_holdout, pred_h),
    })
tabla_holdout = pd.DataFrame(resultados_holdout).sort_values("RMSE_holdout").reset_index(drop=True)
print("Métricas sobre el holdout final (últimos 90 días, no vistos durante entrenamiento/CV):")
tabla_holdout

mejor_modelo_nombre = tabla_holdout.iloc[0]["Modelo"]
mejor_modelo = modelos_entrenados[mejor_modelo_nombre]
print(f"Mejor modelo según RMSE de holdout: {mejor_modelo_nombre}")

**Fase 5 en una línea:** Random Forest fue el mejor de los cuatro sobre el holdout no contaminado, con XGBoost muy cerca; la brecha entre algoritmos es chica frente a la que hay contra los modelos estadísticos de la Fase 4.


## FASE 6 - Deep Learning (LSTM)

Modelo secuencial simple para capturar memoria temporal (día anterior, semana pasada,
mes pasado). Requiere TensorFlow, disponible por defecto en Colab.

**Ajustes de esta fase:**
- **Corrección de fuga de datos:** la versión original ajustaba `MinMaxScaler` sobre toda la
  serie (`X`, `y`) antes de separar train/test, es decir que el escalado ya "conocía" el rango de
  valores del período de test. Ahora el scaler se ajusta (`fit`) solo con el período de
  entrenamiento y se aplica (`transform`) al de test; para no perder continuidad en la ventana de
  secuencias en el borde train/test, se usan las últimas `ventana` filas de train como contexto
  inicial del test.
- Se agrega `EarlyStopping` (con restauración de los mejores pesos) y se sube el techo de épocas
  de 25 a 60, para que el entrenamiento se adapte a cuándo deja de mejorar en validación en lugar
  de correr un número de épocas fijo y arbitrario.

In [ ]:
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping

    tf.random.set_seed(RANDOM_STATE)

    ventana = 30
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()

    corte = int(len(X) * 0.85)
    X_train_raw, X_test_raw = X.iloc[:corte], X.iloc[corte:]
    y_train_raw, y_test_raw = y.iloc[:corte], y.iloc[corte:]

    X_train_scaled = scaler_X.fit_transform(X_train_raw)
    X_test_scaled = scaler_X.transform(X_test_raw)
    y_train_scaled = scaler_y.fit_transform(y_train_raw.values.reshape(-1, 1))
    y_test_scaled = scaler_y.transform(y_test_raw.values.reshape(-1, 1))

    def crear_secuencias(X_arr, y_arr, ventana):
        Xs, ys = [], []
        for i in range(ventana, len(X_arr)):
            Xs.append(X_arr[i-ventana:i])
            ys.append(y_arr[i])
        return np.array(Xs), np.array(ys)

    # Se antepone el final del train (tamaño `ventana`) al test para no perder los primeros
    # `ventana` días del período de test al construir secuencias, sin que el scaler haya visto
    # datos de test en ningún momento.
    X_test_scaled_ext = np.concatenate([X_train_scaled[-ventana:], X_test_scaled], axis=0)
    y_test_scaled_ext = np.concatenate([y_train_scaled[-ventana:], y_test_scaled], axis=0)

    X_train_seq, y_train_seq = crear_secuencias(X_train_scaled, y_train_scaled, ventana)
    X_test_seq, y_test_seq = crear_secuencias(X_test_scaled_ext, y_test_scaled_ext, ventana)

    modelo_lstm = Sequential([
        LSTM(64, activation="tanh", return_sequences=True, input_shape=(ventana, X_train_seq.shape[2])),
        Dropout(0.2),
        LSTM(32, activation="tanh"),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1)
    ])
    modelo_lstm.compile(optimizer="adam", loss="mse")

    parada_temprana = EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)
    historia = modelo_lstm.fit(X_train_seq, y_train_seq, epochs=60, batch_size=64,
                                validation_split=0.1, callbacks=[parada_temprana], verbose=0)

    pred_lstm_scaled = modelo_lstm.predict(X_test_seq, verbose=0)
    pred_lstm = scaler_y.inverse_transform(pred_lstm_scaled).flatten()
    real_lstm = scaler_y.inverse_transform(y_test_seq).flatten()

    rmse_lstm = np.sqrt(mean_squared_error(real_lstm, pred_lstm))
    mae_lstm = mean_absolute_error(real_lstm, pred_lstm)
    print(f"LSTM -> RMSE: {rmse_lstm:.2f} mm | MAE: {mae_lstm:.2f} mm")

    plt.figure(figsize=(12,5))
    plt.plot(real_lstm[-60:], label="Real")
    plt.plot(pred_lstm[-60:], label="Predicción LSTM")
    plt.legend(); plt.title("LSTM - Precipitación diaria, últimos 60 días de test"); plt.show()
except ImportError:
    print("TensorFlow no disponible en este entorno.")
    rmse_lstm, mae_lstm = np.nan, np.nan

**Fase 6 en una línea:** la LSTM mejora a SARIMA/Prophet pero no a los modelos de árboles de la Fase 5; con el volumen de datos disponible, el Deep Learning no aporta ventaja adicional acá.


## FASE 7 - Validación y comparación final de modelos

Se consolidan las métricas de todos los modelos (estadísticos, ML y deep learning) usando
el mismo criterio: **RMSE** (penaliza errores grandes, ideal para eventos de lluvia intensa)
y **MAE** (error promedio, fácil interpretación). Nota: SARIMA/Prophet se evalúan sobre la serie
**semanal** (mm/semana) y ML/LSTM sobre la serie **diaria** (mm/día); las magnitudes no son
directamente comparables entre ambos grupos, sólo el ranking dentro de cada uno.

In [ ]:
comparacion_final = tabla_holdout.copy()
comparacion_final = pd.concat([
    comparacion_final,
    pd.DataFrame([
        {"Modelo": "SARIMA", "RMSE_holdout": rmse_sarima, "MAE_holdout": mae_sarima, "R2_holdout": np.nan},
        {"Modelo": "Prophet", "RMSE_holdout": rmse_prophet, "MAE_holdout": mae_prophet, "R2_holdout": np.nan},
        {"Modelo": "LSTM", "RMSE_holdout": rmse_lstm, "MAE_holdout": mae_lstm, "R2_holdout": np.nan},
    ])
], ignore_index=True).sort_values("RMSE_holdout").reset_index(drop=True)

comparacion_final

**Fase 7 en una línea:** jerarquía final — Random Forest < XGBoost < LightGBM < Gradient Boosting < LSTM < SARIMA ≈ Prophet. Random Forest queda como modelo operativo (Fase 8); SARIMA sigue siendo el que proyecta a futuro (Fase 9).


## FASE 8 - Interpretabilidad del modelo

Se utiliza el mejor modelo de Machine Learning (según RMSE de holdout) para explicar sus
predicciones con SHAP, Permutation Importance y Partial Dependence Plots.

**Ajuste:** se calcula todo sobre `X_holdout` (datos que el modelo no vio en entrenamiento ni en
CV) en lugar de sobre `X` completo. Explicar un modelo sobre los mismos datos con los que se
entrenó tiende a sobreestimar la importancia de las variables, porque el modelo puede haber
memorizado parte de esos datos. Como el modelo predice en escala `log1p`, se usa un envoltorio
que destransforma (`expm1`) para que SHAP/Permutation/PDP trabajen directamente en mm. El
envoltorio hereda de `BaseEstimator`/`RegressorMixin` de scikit-learn (en vez de ser una clase
mínima con solo `predict()`) porque tanto `permutation_importance` como `PartialDependenceDisplay`
validan que el estimador recibido cumpla la API completa de scikit-learn (método `fit`, atributo
`_estimator_type="regressor"` y pasar `check_is_fitted`); una clase sin esos atributos hace que
esas funciones fallen con `InvalidParameterError` antes de llegar a calcular nada.
"""

In [ ]:
from sklearn.base import BaseEstimator, RegressorMixin

class ModeloEscalaOriginal(BaseEstimator, RegressorMixin):
    """Envoltorio sklearn-compatible: expone predict() en mm, ya que el modelo interno
    (self.modelo) fue entrenado sobre log1p(precipitación). No reentrena nada: `fit()` sólo deja
    constancia de que el objeto está "ajustado" para que check_is_fitted lo acepte."""
    def __init__(self, modelo=None):
        self.modelo = modelo

    def fit(self, X, y=None):
        self.fitted_ = True
        return self

    def predict(self, X):
        return np.clip(np.expm1(self.modelo.predict(X)), 0, None)

mejor_modelo_mm = ModeloEscalaOriginal(mejor_modelo).fit(X_holdout, y_holdout)

try:
    import shap
    muestra = X_holdout.sample(min(90, len(X_holdout)), random_state=RANDOM_STATE)
    explainer = shap.Explainer(mejor_modelo_mm.predict, muestra)
    valores_shap = explainer(muestra)

    shap.summary_plot(valores_shap, muestra, show=True)
except Exception as e:
    print(f"No se pudo calcular SHAP para este modelo: {e}")


# %%

resultado_permutacion = permutation_importance(
    mejor_modelo_mm, X_holdout, y_holdout, scoring="r2",
    n_repeats=8, random_state=RANDOM_STATE, n_jobs=-1
)
importancia_df = pd.DataFrame({
    "variable": X_holdout.columns,
    "importancia_media": resultado_permutacion.importances_mean,
    "importancia_std": resultado_permutacion.importances_std
}).sort_values("importancia_media", ascending=False).head(15)

importancia_df = importancia_df.sort_values("importancia_media", ascending=True)

plt.figure(figsize=(9,7))
colores = sns.color_palette("mako", len(importancia_df))
plt.barh(importancia_df["variable"], importancia_df["importancia_media"],
         xerr=importancia_df["importancia_std"], color=colores, capsize=4)
plt.title(f"Permutation Importance (holdout) - {mejor_modelo_nombre}")
plt.xlabel("Disminución de R² al permutar la variable")
plt.tight_layout()
plt.show()


# %%

variables_pdp = ["oni", "humedad", "precip_acum_30", "caudal_parana"]
variables_pdp = [v for v in variables_pdp if v in X_holdout.columns]

# Las versiones nuevas de scikit-learn no aceptan columnas enteras en el PDP: se pasa todo a float.
X_pdp = X_holdout.astype(float)

try:
    fig, ax = plt.subplots(figsize=(13, 8))
    PartialDependenceDisplay.from_estimator(mejor_modelo_mm, X_pdp, variables_pdp, ax=ax)
    plt.suptitle(f"Partial Dependence Plots (holdout) - {mejor_modelo_nombre}")
    plt.tight_layout()
    plt.show()
except Exception as e:
    # Si falla en mm, se grafica sobre el modelo interno (escala log1p) y se aclara en el título.
    plt.close("all")
    print(f"PDP en mm falló ({e}); graficando en escala log1p(mm).")
    try:
        fig, ax = plt.subplots(figsize=(13, 8))
        PartialDependenceDisplay.from_estimator(mejor_modelo, X_pdp, variables_pdp, ax=ax)
        plt.suptitle(f"Partial Dependence Plots (holdout, escala log1p) - {mejor_modelo_nombre}")
        plt.tight_layout()
        plt.show()
    except Exception as e2:
        plt.close("all")
        print(f"No se pudo generar el PDP: {e2}. El resto del análisis continúa.")


**Fase 8 en una línea:** humedad, viento y presión explican casi todo lo que predice Random Forest; el ONI y los rezagos de lluvia pesan menos de lo esperado en el dato diario puntual.


## FASE 9 - Riesgo para mantenimiento preventivo

Se traduce el pronóstico en niveles de riesgo (percentiles de la Fase 3) y se sugiere una reprogramación automática de tareas sensibles al clima, usando `media_futuro` (Fase 4.1.1) — no el período de test. El horizonte cubierto es el que resulte de `n_future` en el momento de ejecutar el notebook.


In [ ]:
def clasificar_riesgo(precip_semanal_predicha):
    if precip_semanal_predicha < UMBRAL_MODERADO:
        return "Normal"
    elif precip_semanal_predicha < UMBRAL_ALTO:
        return "Moderado"
    elif precip_semanal_predicha < UMBRAL_CRITICO:
        return "Alto"
    else:
        return "Crítico"

n_semanas_forecast = n_future
horizonte = media_futuro.index[:n_semanas_forecast]
valores_forecast = media_futuro.reindex(horizonte)

tareas_ejemplo = pd.DataFrame({
    "tarea": ["Inspección de líneas eléctricas exteriores", "Mantenimiento de techos y desagües",
              "Trabajos en altura", "Poda y despeje de vegetación", "Revisión de drenajes pluviales",
              "Pintura y trabajos exteriores", "Inspección de caminos internos", "Mantenimiento de equipos móviles"],
})

reporte_riesgo = pd.DataFrame({
    "semana": horizonte,
    "precipitacion_esperada_mm": np.round(valores_forecast.values, 1),
    "riesgo": [clasificar_riesgo(v) for v in valores_forecast.values]
})

def sugerir_reprogramacion(riesgo):
    return {
        "Normal": "Sin restricciones. Ejecutar según cronograma.",
        "Moderado": "Monitorear pronóstico. Priorizar tareas no críticas primero en la semana.",
        "Alto": "Reprogramar tareas exteriores sensibles al clima a días de menor probabilidad de lluvia.",
        "Crítico": "Posponer tareas exteriores. Reforzar inspección de drenajes y sistemas eléctricos.",
    }[riesgo]

reporte_riesgo["accion_sugerida"] = reporte_riesgo["riesgo"].map(sugerir_reprogramacion)

print(f"Horizonte del reporte de riesgo: {len(reporte_riesgo)} semanas "
      f"({reporte_riesgo['semana'].min().date()} -> {reporte_riesgo['semana'].max().date()})")
print("\nDistribución de niveles de riesgo en el horizonte:")
print(reporte_riesgo["riesgo"].value_counts())
reporte_riesgo

colores_riesgo = {"Normal": "green", "Moderado": "gold", "Alto": "orange", "Crítico": "red"}

# Con varias decenas de semanas, una etiqueta de fecha por barra satura el eje X: se muestra
# ~1 de cada 4 (aprox. mensual), sin dejar de dibujar todas las barras.
plt.figure(figsize=(16,5))
plt.bar(reporte_riesgo["semana"].astype(str), reporte_riesgo["precipitacion_esperada_mm"],
        color=[colores_riesgo[r] for r in reporte_riesgo["riesgo"]])
plt.axhline(UMBRAL_MODERADO, linestyle="--", color="gray", alpha=0.6, label="P50")
plt.axhline(UMBRAL_ALTO, linestyle="--", color="orange", alpha=0.6, label="P90")
plt.axhline(UMBRAL_CRITICO, linestyle="--", color="red", alpha=0.6, label="P95")
paso_etiquetas = max(1, len(reporte_riesgo) // 12)
ax = plt.gca()
ax.set_xticks(range(0, len(reporte_riesgo), paso_etiquetas))
ax.set_xticklabels(reporte_riesgo["semana"].astype(str).iloc[::paso_etiquetas], rotation=45, ha="right")
plt.ylabel("Precipitación esperada (mm/semana)")
plt.title(f"Riesgo semanal de precipitación - Zárate-Lima "
          f"({reporte_riesgo['semana'].min().date()} a {reporte_riesgo['semana'].max().date()})")
plt.legend()
plt.tight_layout()
plt.show()

**Fase 9 en una línea:** cada semana del horizonte pronosticado queda clasificada en Normal / Moderado / Alto / Crítico, con una acción de mantenimiento sugerida para cada nivel; el conteo exacto por nivel lo imprime esta misma celda al ejecutarla.


## Conclusión y próximos pasos

El pipeline corre de punta a punta en Colab con datos reales de ONI (NOAA) y de precipitación/meteorología (Open-Meteo), con la hidrología del Paraná como único tramo aún sintético (el INA no tiene API pública). Se compararon 7 modelos bajo el mismo criterio (RMSE/MAE sobre datos no vistos); Random Forest ganó y es el que se usa para interpretabilidad, mientras SARIMA sostiene la proyección a futuro. La fecha de corte y el horizonte de pronóstico se recalculan solos en cada corrida, así que el reporte de riesgo de la Fase 9 siempre queda referido a "desde hoy en adelante" y no a un rango que ya pasó. La fase ENSO vigente (sección 1.1b) puede cruzarse con [AgroENSO](https://agroenso.netlify.app/) para ver el mismo fenómeno desde el ángulo agrícola.

**Próximos pasos:**
1. Conseguir datos reales de caudal del Paraná (INA/Prefectura) en lugar de la serie sintética.
2. Reemplazar la persistencia del último ONI por el outlook probabilístico de IRI/CPC para el ONI futuro.
3. Automatizar la re-ejecución periódica (semanal o mensual) y el monitoreo de caída de las fuentes Open-Meteo/NOAA.
4. Conectar `reporte_riesgo` a un dashboard para uso de los planificadores de mantenimiento.
